# Phases 8-13: FPMC, Hybrid Fusion, Ablation & Final Evaluation

This comprehensive notebook covers:
- **Phase 8**: Sequential Recommendation (FPMC)
- **Phase 9**: Full Hybrid Integration
- **Phase 10**: Hyperparameter Tuning
- **Phase 11**: Ablation Study
- **Phase 12**: Evaluation & Analysis
- **Phase 13**: Research Validation

**Run Time**: ~30 mins
**Input**: All outputs from Phases 1-7
**Output**: Final models, metrics, ablations, visualizations

In [1]:
# Cell 2: Imports
import os
import pickle
import json
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.preprocessing import normalize
import warnings
warnings.filterwarnings('ignore')

# GPU setup
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

# Environment detection - local vs Kaggle
BASE_DIR = os.getcwd()
if os.path.exists('/kaggle/input/kaggle-phase2-phase3-outputs/train.csv'):
    # Real Kaggle environment
    OUTPUT_DIR = '/kaggle/working'
else:
    # Local environment
    OUTPUT_DIR = os.path.join(BASE_DIR, 'output', 'phase-8-13-results')

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

Device: cuda
OUTPUT_DIR: /kaggle/working/output/phase-8-13-results


In [2]:
# Cell 2: Imports
import os
import pickle
import json
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.preprocessing import normalize
import warnings
warnings.filterwarnings('ignore')

# GPU setup
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

OUTPUT_DIR = '/kaggle/working'
os.makedirs(OUTPUT_DIR, exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

Device: cuda


In [ ]:
# Cell 3: Load all models
print("Loading Phase 1-7 outputs...")

# Detect paths
BASE_DIR = os.getcwd()
if os.path.exists('/kaggle/input/datasets/chandrimanandi/phase-2-3-results'):
    # Real Kaggle environment
    phase23_dir = '/kaggle/input/datasets/chandrimanandi/phase-2-3-results'
    phase4_dir = '/kaggle/input/datasets/chandrimanandi/phase-4-results'
    phase7_dir = '/kaggle/input/datasets/chandrimanandi/phase-7-results'
else:
    # Local environment
    phase23_dir = os.path.join(BASE_DIR, 'output')
    phase4_dir = os.path.join(BASE_DIR, 'output', 'phase-4-results')
    phase7_dir = os.path.join(BASE_DIR, 'output', 'phase-7-results')

# Phase 2-3: Data
train_df = pd.read_csv(os.path.join(phase23_dir, 'train.csv'))
val_df = pd.read_csv(os.path.join(phase23_dir, 'val.csv'))
test_df = pd.read_csv(os.path.join(phase23_dir, 'test.csv'))

with open(os.path.join(phase23_dir, 'id_maps.pkl'), 'rb') as f:
    id_maps = pickle.load(f)

# Phase 4: SVD
with open(os.path.join(phase4_dir, 'biased_svd_model.pkl'), 'rb') as f:
    svd_model = pickle.load(f)

# Phase 4: SVD+ST (removed - using BERT-only system)
# All embedding-based features removed per user directive

# Phase 7: node2vec
with open(os.path.join(phase7_dir, 'n2v_embeddings.pkl'), 'rb') as f:
    n2v_data = pickle.load(f)

n_users = id_maps['n_users']
n_items = id_maps['n_items']
P = svd_model['P']
Q = svd_model['Q']
bu = svd_model['bu']
bi = svd_model['bi']
r_mean = svd_model['global_mean']

n2v_item_emb_norm = n2v_data['item_embeddings_normalized']

print(f"✓ All models loaded")
print(f"  Users: {n_users}, Items: {n_items}")
print(f"  SVD dims: P={P.shape}, Q={Q.shape}")
print(f"  node2vec dims: {n2v_item_emb_norm.shape}")


Loading Phase 1-7 outputs...
✓ All models loaded
  Users: 5541, Items: 3568
  SVD dims: P=(5541, 20), Q=(3568, 20)
  node2vec dims: (3568, 384)


## PHASE 8: Sequential Recommendation (FPMC)

### Hybrid Fusion Architecture:

**Scoring Function** (combines multiple recommendation signals):
```
score(item) = α·SVD(user, item) + β·FPMC(prev_item, item) + γ·KG_similarity(prev_item, item)
```

**Components**:
1. **SVD Term** (α = included):
   - `P[user] @ Q[item].T + bu[user] + bi[item] + global_mean`
   - Captures user preferences and item biases

2. **FPMC Term** (β = learned weight):
   - `(M[user] + N[prev_item]) @ N[item]`
   - Sequential/Markov assumption: prev_item influences next_item
   - Personalized by user

3. **KG Similarity Term** (γ = learned weight):
   - `cosine(KG_emb[prev_item], KG_emb[item])`
   - KG embeddings from node2vec (Phase 7)
   - Contains both semantic (reviews) + structural (metadata) info
   - Items sharing: same genre, brand, category → high similarity
   - Helps with cold-start: recommend similar items even without interaction

**Learning Strategy**:
- SVD: Pre-trained (Phase 4)
- FPMC: Trained with SGD on sequential triples
- KG weights (β, γ): Tuned on validation set to maximize NDCG@10

In [4]:
# Cell 4: FPMC Model
print("\n" + "="*60)
print("PHASE 8: FPMC - SEQUENTIAL RECOMMENDATION")
print("="*60)

class FPMC(nn.Module):
    """Factorizing Personalized Markov Chains"""
    def __init__(self, n_users, n_items, k_factors):
        super().__init__()
        self.n_users = n_users
        self.n_items = n_items  
        self.k = k_factors
        
        # User sequential embeddings
        self.M = nn.Parameter(torch.randn(n_users, k_factors) * 0.01)
        # Item sequential embeddings
        self.N = nn.Parameter(torch.randn(n_items, k_factors) * 0.01)
    
    def forward(self, user_idx, prev_item_idx, target_item_idx):
        """
        Args:
            user_idx: (batch,) user indices
            prev_item_idx: (batch,) previous item indices
            target_item_idx: (batch,) target item indices
        Returns:
            scores: (batch,) pairwise scores
        """
        m_u = self.M[user_idx]  # (batch, k)
        n_prev = self.N[prev_item_idx]  # (batch, k)
        n_target = self.N[target_item_idx]  # (batch, k)
        
        # Transition: (m_u + n_prev) · n_target
        score = ((m_u + n_prev) * n_target).sum(dim=1)
        return score

K_FPMC = P.shape[1]
fpmc_model = FPMC(n_users, n_items, K_FPMC).to(device)
print(f"FPMC model initialized with k={K_FPMC}")


PHASE 8: FPMC - SEQUENTIAL RECOMMENDATION
FPMC model initialized with k=20


In [5]:
# Cell 5: Prepare FPMC training data
print("Preparing FPMC training data...")

# Sort by user and timestamp
train_sorted = train_df.sort_values(['user_idx', 'timestamp']).reset_index(drop=True)

# Create (user, prev_item, next_item) triples
fpmc_triples = []
for user_id in tqdm(range(n_users), desc="Creating FPMC triples"):
    user_seq = train_sorted[train_sorted['user_idx'] == user_id]
    if len(user_seq) >= 2:
        items = user_seq['item_idx'].values.astype(int)
        # Create consecutive pairs
        for i in range(len(items) - 1):
            prev_item = int(items[i])
            next_item = int(items[i+1])
            fpmc_triples.append((int(user_id), prev_item, next_item))

print(f"FPMC triples: {len(fpmc_triples)}")

# Convert to tensors
if fpmc_triples:
    fpmc_triples = np.array(fpmc_triples)
    fpmc_users = torch.LongTensor(fpmc_triples[:, 0]).to(device)
    fpmc_prev = torch.LongTensor(fpmc_triples[:, 1]).to(device)
    fpmc_next = torch.LongTensor(fpmc_triples[:, 2]).to(device)
    print(f"Tensors created: users={fpmc_users.shape}, prev={fpmc_prev.shape}, next={fpmc_next.shape}")

Preparing FPMC training data...


Creating FPMC triples: 100%|██████████| 5541/5541 [00:01<00:00, 2950.66it/s]

FPMC triples: 48083
Tensors created: users=torch.Size([48083]), prev=torch.Size([48083]), next=torch.Size([48083])


In [6]:
# Cell 6: Train FPMC
if len(fpmc_triples) > 0:
    print("\nTraining FPMC...")
    
    criterion = nn.MSELoss()
    optimizer = torch.optim.SGD(fpmc_model.parameters(), lr=0.01, weight_decay=0.001)
    
    FPMC_EPOCHS = 20
    FPMC_BATCH = 256
    
    for epoch in range(FPMC_EPOCHS):
        fpmc_model.train()
        
        # Shuffle
        perm = torch.randperm(len(fpmc_users))
        users_shuffled = fpmc_users[perm]
        prev_shuffled = fpmc_prev[perm]
        next_shuffled = fpmc_next[perm]
        
        epoch_loss = 0
        n_batches = 0
        
        for i in range(0, len(fpmc_users), FPMC_BATCH):
            end = min(i + FPMC_BATCH, len(fpmc_users))
            u_batch = users_shuffled[i:end]
            prev_batch = prev_shuffled[i:end]
            next_batch = next_shuffled[i:end]
            
            optimizer.zero_grad()
            scores = fpmc_model(u_batch, prev_batch, next_batch)
            loss = criterion(scores, torch.ones_like(scores))
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            n_batches += 1
        
        if (epoch + 1) % 5 == 0:
            avg_loss = epoch_loss / n_batches
            print(f"Epoch {epoch+1:2d} | Loss: {avg_loss:.4f}")
    
    # Extract FPMC matrices
    M = fpmc_model.M.detach().cpu().numpy()
    N = fpmc_model.N.detach().cpu().numpy()
    print(f"✓ FPMC trained: M={M.shape}, N={N.shape}")
else:
    print("No FPMC triples - skipping FPMC training")
    M = np.random.randn(n_users, K_FPMC) * 0.01
    N = np.random.randn(n_items, K_FPMC) * 0.01


Training FPMC...
Epoch  5 | Loss: 1.0000
Epoch 10 | Loss: 1.0000
Epoch 15 | Loss: 1.0000
Epoch 20 | Loss: 1.0000
✓ FPMC trained: M=(5541, 20), N=(3568, 20)


## PHASE 9: Hybrid Fusion & PHASE 10: Tuning

In [7]:
# Cell 7: Get last interaction per user
print("\n" + "="*60)
print("PHASE 9: HYBRID FUSION")
print("PHASE 10: HYPERPARAMETER TUNING")
print("="*60)

# Extract previous item per user
if 'timestamp' in train_df.columns:
    last_item = train_df.sort_values('timestamp').groupby('user_idx')['item_idx'].last()
else:
    last_item = train_df.groupby('user_idx')['item_idx'].last()

print(f"Last items extracted for {len(last_item):,} users")


PHASE 9: HYBRID FUSION
PHASE 10: HYPERPARAMETER TUNING
Last items extracted for 5,541 users


In [ ]:
# Cell 8: Hybrid scoring function (with normalisation)
print("\n" + "="*60)
print("HYBRID SCORING FUNCTION")
print("="*60)

def normalise(arr):
    """Zero-mean unit-std normalisation. Returns zeros if flat."""
    std = arr.std()
    return (arr - arr.mean()) / std if std > 1e-9 else np.zeros_like(arr)

def score_all_items_hybrid(user_idx, prev_item_idx, w_fpmc=0.3, w_kg=0.3):
    """
    Compute hybrid scores using BERT-only SVD (no Sentence Transformers).
    
    Args:
        user_idx: User index
        prev_item_idx: Previous item index (for sequential)
        w_fpmc: Weight for FPMC sequential component
        w_kg: Weight for Knowledge Graph component
    """
    svd_scores  = np.zeros(n_items)
    fpmc_scores = np.zeros(n_items)
    kg_scores   = np.zeros(n_items)

    # Standard SVD with BERT sentiment (no Sentence Transformers)
    if 0 <= user_idx < P.shape[0]:
        svd_scores = P[user_idx] @ Q.T + bu[user_idx] + bi + r_mean

    # FPMC
    if prev_item_idx is not None and 0 <= user_idx < M.shape[0] and 0 <= prev_item_idx < N.shape[0]:
        fpmc_scores = (M[user_idx] + N[prev_item_idx]) @ N.T

    # KG
    if prev_item_idx is not None and 0 <= prev_item_idx < n2v_item_emb_norm.shape[0]:
        kg_scores = n2v_item_emb_norm[prev_item_idx] @ n2v_item_emb_norm.T

    # Normalise to equal scale so no single term dominates
    svd_norm  = normalise(svd_scores)
    fpmc_norm = normalise(fpmc_scores)
    kg_norm   = normalise(kg_scores)

    hybrid = svd_norm + w_fpmc * fpmc_norm + w_kg * kg_norm
    return hybrid, svd_norm, fpmc_norm, kg_norm

print("Hybrid scoring function defined (BERT-only: per-component normalisation)")



HYBRID SCORING FUNCTION
Hybrid scoring function defined (with per-component normalisation)


In [9]:
# ── Scoring sanity check ──────────────────────────────────────────────────────
print("=== SCORING SANITY CHECK ===")

sample_uid  = int(test_df['user_idx'].iloc[0])
sample_prev = int(last_item.get(sample_uid, 0))

h, svd_n, fpmc_n, kg_n = score_all_items_hybrid(sample_uid, sample_prev, w_fpmc=0.3, w_kg=0.3)

print(f"User {sample_uid}, prev_item {sample_prev}:")
print(f"  SVD  component std: {svd_n.std():.6f}")
print(f"  FPMC component std: {fpmc_n.std():.6f}")
print(f"  KG   component std: {kg_n.std():.6f}")
print(f"  Hybrid scores std:  {h.std():.6f}")
print(f"  Top-3 items: {np.argsort(h)[::-1][:3].tolist()}")

# All three should have std ≈ 1.0 after normalisation (or 0 if disabled)
for name, arr in [('SVD', svd_n), ('FPMC', fpmc_n), ('KG', kg_n)]:
    status = "✅" if abs(arr.std() - 1.0) < 0.01 or arr.std() == 0 else "⚠️ "
    print(f"  {status} {name} std={arr.std():.4f} ({'OK' if arr.std() > 0 else 'disabled/flat'})")

=== SCORING SANITY CHECK ===
User 0, prev_item 1164:
  SVD  component std: 1.000000
  FPMC component std: 1.000000
  KG   component std: 1.000000
  Hybrid scores std:  1.075082
  Top-3 items: [1164, 2681, 3398]
  ✅ SVD std=1.0000 (OK)
  ✅ FPMC std=1.0000 (OK)
  ✅ KG std=1.0000 (OK)


In [ ]:
# Cell 9: Evaluation metric
def ndcg_at_k(rank, k=10):
    """Calculate NDCG@k from 0-indexed rank."""
    if rank < k:
        return 1.0 / np.log2(rank + 2)
    return 0.0

def evaluate_model(eval_df, w_fpmc=0.3, w_kg=0.3, k=10):
    """
    Evaluate hybrid model on dataset.
    
    Computes NDCG@10 - how well top-10 recommendations rank the held-out item.
    
    Args:
        eval_df: Dataset with columns [user_idx, item_idx, ...]
        w_fpmc: Weight for sequential (FPMC) component (learned)
        w_kg: Weight for knowledge graph similarity component (learned)
        k: Cutoff for NDCG computation
    
    Returns:
        Mean NDCG@k across all items in eval_df
    """
    ndcgs = []
    
    for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), leave=False):
        uid = int(row['user_idx'])
        iid = int(row['item_idx'])
        
        if iid >= n_items or iid < 0:
            continue
        
        # Get previous item (from training data)
        prev_iid = last_item.get(uid, None)
        if prev_iid is not None:
            prev_iid = int(prev_iid)
            if prev_iid >= n_items:
                prev_iid = None
        
        # Compute hybrid scores
        scores, _, _, _ = score_all_items_hybrid(uid, prev_iid, w_fpmc, w_kg)
        
        # Get rank of held-out item
        rank = int(np.sum(scores > scores[iid]))
        ndcgs.append(ndcg_at_k(rank, k))
    
    return float(np.mean(ndcgs)) if ndcgs else 0.0

print("Evaluation function ready (NDCG@10 metric)")
print("\nEvaluation metric:")
print("  NDCG@10: Normalized Discounted Cumulative Gain")
print("  Measures ranking quality of held-out items in top-10 predictions")


Evaluation function ready (NDCG@10 metric)

Evaluation metric:
  NDCG@10: Normalized Discounted Cumulative Gain
  Measures ranking quality of held-out items in top-10 predictions


In [11]:
# Cell 10: Tune KG weight
print("\nTuning KG weight on validation set...")

w_kg_values = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
val_ndcgs = []

for w_kg in w_kg_values:
    ndcg = evaluate_model(val_df, w_fpmc=0.3, w_kg=w_kg, k=10)
    val_ndcgs.append(ndcg)
    print(f"  w_kg={w_kg:.1f} → NDCG@10={ndcg:.4f}")

best_w_kg = w_kg_values[np.argmax(val_ndcgs)]
print(f"\nBest w_kg: {best_w_kg}")

print("\n" + "="*60)
print("WHY KG HELPS - THE COLD-START PROBLEM")
print("="*60)
print("""
Traditional Sequential Recommendation (FPMC only):
- Needs item co-purchase history: prev_item → next_item
- Cold-start: New items with no history can't be recommended well
- Problem: Sparse interaction data limits what can be learned

Knowledge Graph Solution:
- Items related by metadata (category, brand, genre, artist)
- Even without prior interaction, KG similarity is high
- Example: Two songs by same artist/genre
  - May never co-appear in sequences
  - But KG embeddings learned to be similar
  - Can be recommended based on structural similarity
  
Result: w_kg = {best_w_kg:.1f}
- This weight balances: interaction patterns + structural similarity
- Helps recommend similar items even without co-purchase history
- Solves sparsity in sequential data
""".format(best_w_kg=best_w_kg))


Tuning KG weight on validation set...


  w_kg=0.0 → NDCG@10=0.0063


  w_kg=0.1 → NDCG@10=0.0066


  w_kg=0.2 → NDCG@10=0.0068


  w_kg=0.3 → NDCG@10=0.0068


  w_kg=0.4 → NDCG@10=0.0074


  w_kg=0.5 → NDCG@10=0.0083

Best w_kg: 0.5

WHY KG HELPS - THE COLD-START PROBLEM

Traditional Sequential Recommendation (FPMC only):
- Needs item co-purchase history: prev_item → next_item
- Cold-start: New items with no history can't be recommended well
- Problem: Sparse interaction data limits what can be learned

Knowledge Graph Solution:
- Items related by metadata (category, brand, genre, artist)
- Even without prior interaction, KG similarity is high
- Example: Two songs by same artist/genre
  - May never co-appear in sequences
  - But KG embeddings learned to be similar
  - Can be recommended based on structural similarity
  
Result: w_kg = 0.5
- This weight balances: interaction patterns + structural similarity
- Helps recommend similar items even without co-purchase history
- Solves sparsity in sequential data



## PHASE 11: Ablation Study

In [ ]:
# Cell 11: Ablation configurations
print("\n" + "="*60)
print("PHASE 11: ABLATION STUDY")
print("="*60)

print("""
Ablation Study: Isolate contribution of each component

This validates that each component improves performance:
1. Does SVD benefit from Sentence Transformer embeddings?
2. Do we need FPMC (sequential)?
3. Do we need KG (semantic similarity)?
4. Does combining all help?
""")

ablation_configs = [
    {
        'name': 'SVD only',           
        'w_fpmc': 0.0, 
        'w_kg': 0.0,
        'desc': 'Base: Matrix factorization with BERT sentiment (user preference + item bias)'
    },
    {
        'name': 'SVD + FPMC',         
        'w_fpmc': 0.3, 
        'w_kg': 0.0,
        'desc': 'Sequential: Adds previous item influence'
    },
    {
        'name': 'SVD + KG',           
        'w_fpmc': 0.0, 
        'w_kg': best_w_kg,
        'desc': 'Metadata: Adds semantic/structural similarity from KG'
    },
    {
        'name': 'SVD + FPMC + KG',    
        'w_fpmc': 0.3, 
        'w_kg': best_w_kg,
        'desc': 'Hybrid: All components combined - full BERT-only system'
    },
]

print("\nRunning ablation study on test set...\n")

ablation_results = {}
for cfg in ablation_configs:
    ndcg = evaluate_model(test_df, w_fpmc=cfg['w_fpmc'], w_kg=cfg['w_kg'], k=10)
    ablation_results[cfg['name']] = ndcg
    improvement = ndcg - ablation_results.get('SVD only', ndcg)
    print(f"  {cfg['name']:<27} NDCG@10 = {ndcg:.4f} (+{improvement:+.4f} vs SVD)")
    print(f"    → {cfg['desc']}")

print(f"\nBest model: {max(ablation_results, key=ablation_results.get)}")
best_improvement = max(ablation_results.values()) - ablation_results.get('SVD only', 0)
print(f"Improvement over SVD: {best_improvement:+.4f}")


=== DIAGNOSTIC ===
Test users with prev_item: 5541/5541 (100.0%)

For user 0 with prev_item=None:
  SVD  scores — std: 1.000000, min: -4.6585, max: 3.6234
  FPMC scores — std: 0.000000  (should be 0 if prev_item=None)
  KG   scores — std: 0.000000  (should be 0 if prev_item=None)

For user 0 WITH prev_item=1164:
  SVD  scores — std: 1.000000
  FPMC scores — std: 1.000000  (non-zero = FPMC is working)
  KG   scores — std: 1.000000  (non-zero = KG is working)


In [13]:
print(f"r_mean: {r_mean:.4f}")
print(f"bu range: {bu.min():.4f} to {bu.max():.4f},  std: {bu.std():.4f}")
print(f"bi range: {bi.min():.4f} to {bi.max():.4f},  std: {bi.std():.4f}")
print(f"P @ Q.T range for user 0: {(P[0] @ Q.T).min():.4f} to {(P[0] @ Q.T).max():.4f},  std: {(P[0] @ Q.T).std():.6f}")
print(f"\nIf P@Q.T std is tiny (~0.001), the latent factors collapsed during training")

# Also check FPMC scale
sample_prev = int(last_item[0])
fpmc_raw = (M[0] + N[sample_prev]) @ N.T
print(f"\nFPMC raw scores std: {fpmc_raw.std():.6f}")
print(f"KG raw scores std:   {(n2v_item_emb_norm[sample_prev] @ n2v_item_emb_norm.T).std():.6f}")

# Check what w_fpmc and w_kg would need to be to matter
print(f"\nTo match SVD std ({(P[0]@Q.T).std():.6f}), FPMC weight needs to be: {(P[0]@Q.T).std()/max(fpmc_raw.std(),1e-9):.2f}x")

r_mean: 4.2210
bu range: -0.1638 to 0.1529,  std: 0.0835
bi range: -0.2947 to 0.2647,  std: 0.0725
P @ Q.T range for user 0: -0.3105 to 0.1324,  std: 0.026812

If P@Q.T std is tiny (~0.001), the latent factors collapsed during training

FPMC raw scores std: 0.000658
KG raw scores std:   0.069706

To match SVD std (0.026812), FPMC weight needs to be: 40.75x


In [14]:
print(f"P norm (should be non-zero): {np.linalg.norm(P):.6f}")
print(f"Q norm (should be non-zero): {np.linalg.norm(Q):.6f}")
print(f"P max absolute value: {np.abs(P).max():.8f}")
print(f"Q max absolute value: {np.abs(Q).max():.8f}")

P norm (should be non-zero): 24.999151
Q norm (should be non-zero): 31.548433
P max absolute value: 0.26011711
Q max absolute value: 1.09912944


In [ ]:
# Cell 11: Ablation configurations
print("\n" + "="*60)
print("PHASE 11: ABLATION STUDY")
print("="*60)

print("""
Ablation Study: Isolate contribution of each component

This validates that each component improves performance:
1. Do we need FPMC (sequential)?
2. Do we need KG (semantic similarity)?
3. Does combining all help?
""")

ablation_configs = [
    {
        'name': 'SVD only',           
        'w_fpmc': 0.0, 
        'w_kg': 0.0,
        'use_st': False,
        'desc': 'Base: Matrix factorization (user preference + item bias)'
    },
    {
        'name': 'SVD + Sentence Transformers',
        'w_fpmc': 0.0,
        'w_kg': 0.0,
        'use_st': True,
        'desc': 'SVD enhanced with Sentence Transformer semantic embeddings'
    },
    {
        'name': 'SVD + FPMC',         
        'w_fpmc': 0.3, 
        'w_kg': 0.0,
        'use_st': False,
        'desc': 'Sequential: Adds previous item influence'
    },
    {
        'name': 'SVD + KG',           
        'w_fpmc': 0.0, 
        'w_kg': best_w_kg,
        'use_st': False,
        'desc': 'Metadata: Adds semantic/structural similarity from KG'
    },
    {
        'name': 'SVD + FPMC + KG',    
    ndcg = evaluate_model(test_df, w_fpmc=cfg['w_fpmc'], w_kg=cfg['w_kg'], k=10, use_st=cfg.get('use_st', False))
        'w_kg': best_w_kg,
        'use_st': False,
    print(f"  {cfg['name']:<27} NDCG@10 = {ndcg:.4f} (+{improvement:+.4f} vs SVD)")
    },
]

print("\nRunning ablation study on test set...\n")


ablation_results = {}print(f"Improvement over SVD: {best_improvement:+.4f}")

for cfg in ablation_configs:best_improvement = max(ablation_results.values()) - ablation_results.get('SVD only', 0)

    ndcg = evaluate_model(test_df, w_fpmc=cfg['w_fpmc'], w_kg=cfg['w_kg'], k=10)print(f"\nBest model: {max(ablation_results, key=ablation_results.get)}")

    ablation_results[cfg['name']] = ndcg

    improvement = ndcg - ablation_results.get('SVD only', ndcg)    print(f"    → {cfg['desc']}")
    print(f"  {cfg['name']:<20} NDCG@10 = {ndcg:.4f} (+{improvement:+.4f} vs SVD)")


PHASE 11: ABLATION STUDY

Ablation Study: Isolate contribution of each component

This validates that each component improves performance:
1. Do we need FPMC (sequential)?
2. Do we need KG (semantic similarity)?
3. Does combining all help?


Running ablation study on test set...



  SVD only             NDCG@10 = 0.0073 (++0.0000 vs SVD)
    → Base: Matrix factorization (user preference + item bias)


  SVD + FPMC           NDCG@10 = 0.0067 (+-0.0006 vs SVD)
    → Sequential: Adds previous item influence


  SVD + KG             NDCG@10 = 0.0077 (++0.0005 vs SVD)
    → Metadata: Adds semantic/structural similarity from KG


  SVD + FPMC + KG      NDCG@10 = 0.0075 (++0.0002 vs SVD)
    → Hybrid: All components combined - full system

Best model: SVD + KG
Improvement over SVD: +0.0005


## PHASE 12-13: Full Evaluation & Analysis

In [16]:
# Cell 12: Cross-split evaluation
print("\n" + "="*60)
print("PHASE 12-13: FINAL EVALUATION & ANALYSIS")
print("="*60)

print("\nEvaluating on all splits...")
splits = {'train': train_df, 'val': val_df, 'test': test_df}
split_results = {}

for split_name, split_df in splits.items():
    ndcg = evaluate_model(split_df, w_fpmc=0.3, w_kg=best_w_kg, k=10)
    split_results[split_name] = ndcg
    print(f"  {split_name.upper()}: {ndcg:.4f}")


PHASE 12-13: FINAL EVALUATION & ANALYSIS

Evaluating on all splits...


  TRAIN: 0.1136


  VAL: 0.0083


  TEST: 0.0075


In [17]:
# Cell 13: Visualization
print("\nGenerating visualizations...")

# Ablation bar chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Ablation
names = list(ablation_results.keys())
ndcgs = list(ablation_results.values())
colors = ['#c0c0c0', '#90b4ce', '#90ce90', '#2196F3']
ax1.bar(names, ndcgs, color=colors, edgecolor='k', linewidth=0.8)
for i, (name, ndcg) in enumerate(zip(names, ndcgs)):
    ax1.text(i, ndcg + 0.001, f'{ndcg:.4f}', ha='center', va='bottom', fontsize=9)
ax1.set_ylabel('NDCG@10', fontsize=11)
ax1.set_title('Ablation Study', fontsize=12)
ax1.tick_params(axis='x', rotation=45)
ax1.grid(axis='y', alpha=0.3)

# KG tuning
ax2.plot(w_kg_values, val_ndcgs, marker='o', linewidth=2, color='steelblue')
ax2.axvline(best_w_kg, linestyle='--', color='red', alpha=0.7)
ax2.set_xlabel('KG Weight', fontsize=11)
ax2.set_ylabel('NDCG@10', fontsize=11)
ax2.set_title('KG Weight Tuning', fontsize=12)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/final_results.png', dpi=150, bbox_inches='tight')
plt.close()
print("✓ Saved final_results.png")


Generating visualizations...
✓ Saved final_results.png


In [18]:
# Cell 14: Save final results
print("\nSaving final results...")

# Results summary
results = {
    'ablation_study': ablation_results,
    'split_performance': split_results,
    'best_w_kg': float(best_w_kg),
    'kg_tuning_curve': dict(zip([float(w) for w in w_kg_values], [float(n) for n in val_ndcgs])),
}

with open(f'{OUTPUT_DIR}/final_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("✓ Saved final_results.json")

# Results table
results_df = pd.DataFrame({
    'Model': list(ablation_results.keys()),
    'NDCG@10': [f"{v:.4f}" for v in ablation_results.values()],
    'Improvement over SVD': [f"{v - list(ablation_results.values())[0]:+.4f}" for v in ablation_results.values()]
})

results_df.to_csv(f'{OUTPUT_DIR}/results_summary.csv', index=False)
print("✓ Saved results_summary.csv")

print(f"\n{'='*60}")
print("ALL PHASES COMPLETE")
print(f"{'='*60}")
print(f"\nFinal Results:")
print(results_df.to_string(index=False))
print(f"\nFiles saved to: {OUTPUT_DIR}")


Saving final results...
✓ Saved final_results.json
✓ Saved results_summary.csv

ALL PHASES COMPLETE

Final Results:
          Model NDCG@10 Improvement over SVD
       SVD only  0.0073              +0.0000
     SVD + FPMC  0.0067              -0.0006
       SVD + KG  0.0077              +0.0005
SVD + FPMC + KG  0.0075              +0.0002

Files saved to: /kaggle/working
